# DiatomCascadeNet clean reproducible run
This output-free notebook rebuilds the dataset from private inputs, freezes train/validation/test manifests, trains all seven models under one canonical architecture package, and writes every new artifact to an isolated run directory.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), (
    'Run this notebook from the cloned DiatomCascadeNet repository root'
)
os.environ['DIATOM_PROJECT_ROOT'] = str(PROJECT_ROOT)

RUN_ID = 'clean_2026_r01'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'runs' / RUN_ID
os.environ['DIATOM_OUTPUT_DIR'] = str(OUTPUT_DIR)

required = [
    PROJECT_ROOT / 'dataset' / 'raw' / 'labels.csv',
    PROJECT_ROOT / 'dataset' / 'raw' / 'images',
    PROJECT_ROOT / 'dataset' / 'exclusions' / 'invalid_images.csv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing private inputs: {missing}'
assert not OUTPUT_DIR.exists(), f'Choose a new RUN_ID; output already exists: {OUTPUT_DIR}'
print(f'Project: {PROJECT_ROOT}')
print(f'Run: {RUN_ID}')
print(f'Output: {OUTPUT_DIR}')


## 1. Install the package and record the execution environment


In [ ]:
%pip install -q -e .


In [ ]:
import hashlib
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import PIL
import sklearn
import timm
import torch
import torchvision

from diatom_cascade.checkpoints import CHECKPOINT_SCHEMA_VERSION
from diatom_cascade.config.train_and_val_config import TrainAndValConfig

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def run_module(module, *args):
    subprocess.run(
        [sys.executable, '-m', module, *map(str, args)],
        cwd=PROJECT_ROOT,
        check=True,
    )

git_status = subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True
)
assert not git_status.strip(), 'Commit tracked source changes before starting a reproducible run'

training_config = {
    'random_seed': TrainAndValConfig.RANDOM_SEED,
    'image_size': TrainAndValConfig.IMAGE_SIZE,
    'batch_size': TrainAndValConfig.BATCH_SIZE,
    'base_model': TrainAndValConfig.BASE_MODEL,
    'backbone_pretrain': TrainAndValConfig.BACKBONE_PRETRAIN,
    'optimizer': TrainAndValConfig.OPTIMIZER,
    'weight_decay': TrainAndValConfig.WEIGHT_DECAY,
    'max_epochs': TrainAndValConfig.MAX_EPOCHS,
    'initial_lr': TrainAndValConfig.INITIAL_LR,
    'lr_scheduler_type': TrainAndValConfig.LR_SCHEDULER_TYPE,
    'lr_scheduler_config': TrainAndValConfig.LR_SCHEDULER_CONFIG,
    'early_stopping_patience': TrainAndValConfig.EARLY_STOPPING_PATIENCE,
    'early_stopping_min_delta': TrainAndValConfig.EARLY_STOPPING_MIN_DELTA,
    'early_stopping_mode': TrainAndValConfig.EARLY_STOPPING_MODE,
    'focal_alpha': TrainAndValConfig.FOCAL_ALPHA,
    'focal_gamma': TrainAndValConfig.FOCAL_GAMMA,
    'loss_weights': TrainAndValConfig.LOSS_WEIGHTS,
    'augmentation': TrainAndValConfig.AUGMENTATION,
    'normalization': TrainAndValConfig.NORMALIZE,
    'minimum_samples': TrainAndValConfig.MIN_SAMPLES,
}

environment = {
    'run_id': RUN_ID,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'pytorch': torch.__version__,
    'torchvision': torchvision.__version__,
    'timm': timm.__version__,
    'scikit_learn': sklearn.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'pillow': PIL.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cudnn': torch.backends.cudnn.version(),
    'checkpoint_schema_version': CHECKPOINT_SCHEMA_VERSION,
    'training_config': training_config,
    'git_commit': subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True
    ).strip(),
}
environment


## 2. Rebuild and freeze the clean data artifacts
The overwrite flag is used here only before training begins. Do not recreate manifests after the first model starts.

In [ ]:
run_module('unittest', 'discover', '-s', 'tests', '-p', 'test_*.py', '-v')
run_module('scripts.data.cleaning.clean_data')
run_module('scripts.data.preprocessing.build_taxonomy_tree')
run_module('scripts.data.preprocessing.create_filtered_datasets')
run_module('scripts.data.preprocessing.create_split_manifests', '--overwrite')


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
evidence_files = [
    PROJECT_ROOT / 'dataset' / 'exclusions' / 'invalid_images.csv',
    PROJECT_ROOT / 'dataset' / 'cleaned' / 'labels_clean.csv',
    *sorted((PROJECT_ROOT / 'dataset' / 'preprocessed').glob('*.csv')),
    *sorted((PROJECT_ROOT / 'dataset' / 'preprocessed').glob('*.json')),
    *sorted((PROJECT_ROOT / 'dataset' / 'splits').rglob('*.csv')),
]
environment['input_hashes'] = {
    str(path.relative_to(PROJECT_ROOT)): sha256(path) for path in evidence_files
}
with (OUTPUT_DIR / 'run_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(environment, handle, indent=2, ensure_ascii=True)
print(f'Frozen {len(evidence_files)} evidence hashes')

## 3. Train
The four hierarchical stages use the same canonical implementation from src/diatom_cascade/models.py. Each stage initializes new classifier heads and strictly transfers only backbone.* parameters from the immediately preceding checkpoint. F-C, F-G, and F-S are independent ImageNet-initialized baselines.


In [ ]:
training_modules = [
    'scripts.train.train_F_C',
    'scripts.train.train_H_CO',
    'scripts.train.train_H_COF',
    'scripts.train.train_H_COFG',
    'scripts.train.train_H_COFGS',
    'scripts.train.train_F_G',
    'scripts.train.train_F_S',
]
for module in training_modules:
    print(f'\n=== {module} ===', flush=True)
    run_module(module)


## 4. Evaluate the frozen checkpoints
Greedy hierarchical prediction is the primary result. Beam-search numbers remain excluded until the corrected implementation has an independent reference test.

In [ ]:
run_module('scripts.evaluate.run_all_evaluations')


In [ ]:
artifacts = sorted(path for path in OUTPUT_DIR.rglob('*') if path.is_file())
artifact_hashes = {
    str(path.relative_to(OUTPUT_DIR)): sha256(path) for path in artifacts
}
with (OUTPUT_DIR / 'artifact_hashes.json').open('w', encoding='utf-8') as handle:
    json.dump(artifact_hashes, handle, indent=2, ensure_ascii=True)
print(f'Hashed {len(artifacts)} run artifacts in {OUTPUT_DIR}')